# Analyse des déterminants de l'équipement en Véhicules Électriques

## 1. Introduction

Ce projet vise à comprendre quels facteurs (socio-économiques, infrastructures existantes) influencent le taux d'équipement en véhicules électriques au niveau communal en France.

Nous fusionnons trois sources de données provenant de data.gouv.fr :
- IRVE : Données sur les infrastructures de recharge.
- INSEE : Revenus et données socio-économiques.
- Immatriculations : Parc de VE par commune.

## 2. Chargement et Exploration des Données Brutes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loading import load_all_datasets, charger_communes
from src.cleaning import imputer_valeurs_manquantes_fusion, reparer_encodage_implantation, regrouper_implantation_station, standardize_all_codes, clean_irve_variables_finales, corriger_codes_incoherents, corriger_par_nom, corriger_conflit_code_postal, garder_derniere_observation_commune
from src.features import creer_features_irve, creer_taux_equipement_ve
from src.modelisation import modelisation_lasso, modelisation_xgboost
from src.utils import creer_gdf_irve, joindre_communes, ajouter_codes_geo

# Configuration des chemins
PATHS = {
    'irve': 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435',
    'revenu': 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',
    've': 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
}

# Chargement
df_irve_raw, df_revenu_raw, df_ve_raw = load_all_datasets(PATHS)

#### IRVE

In [3]:
print(f"Shape : {df_irve_raw.shape}")
df_irve_raw.sample(5)

Shape : (217227, 52)


,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
163857,CHERBOURG EN COTENTIN,200056844.0,mairie@cherbourg.fr,Zunder (Grupo Easycharger S.A),e-charge50@sdem50.fr,0187650079,e-charge50,FRS50P50173003,NaN,CHERBOURG EN COTENTIN (EQUEURDREVILLE HAINNEVI...,...,25c7d722-c9fd-475a-95fb-f5697893e3a9,e-charge50,2026-03-16T14:32:29.148000+00:00,-1.664315,49.641950,NaN,NaN,False,False,False
205050,ENGIE Vianeo,909073363.0,support.vianeo@engie.com,ENGIE Vianeo,support.vianeo@engie.com,tel:+33-9-69-37-60-09,ENGIE Vianeo,FRVIAP122049,122049,ENGIE Vianeo - B&B HOTEL AVIGNON 2,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,4.872212,43.976953,84130.0,Le Pontet,True,True,False
123703,Power Dot France,891118473.0,hello@powerdot.fr,Power Dot France,hello@powerdot.fr,tel:+33-1-76-31-06-84,Power Dot France,FRPD1PELCMTB,NaN,E.Leclerc - Montauban,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,1.355941,44.004763,82000.0,Montauban,True,True,False
138340,GROUPE INDIGO,NaN,NaN,GROUPE INDIGO,support.emobility.fr@group-indigo.com,NaN,Réseau de recharge du groupe Indigo,FRPKGP75117003,NaN,Porte Maillot,...,d445f73a-797f-46ea-98cc-2927ec7a018d,indigo-group,2025-08-13T13:39:08.570000+00:00,2.281691,48.878800,NaN,NaN,False,False,False
35006,406__eborn,NaN,NaN,SPBR1 | FR*EBN,roaming@freshmile.com,NaN,Réseau eborn,FREBNP8509337084222864581,1399816,Réseau eborn/eaf02376-81e5-5a30-b750-e76a17aa6feb,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,4.673463,44.828462,NaN,NaN,False,False,False


Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).

In [4]:
print(f"Le dataset contient {df_irve_raw.shape[0]} lignes pour {df_irve_raw.shape[1]} variables.")

Le dataset contient 217227 lignes pour 52 variables.


#### Véhicules

In [3]:
print(f"Shape : {df_ve_raw.shape}")
df_ve_raw.sample(5)

Shape : (703545, 8)


,CODGEO,LIBGEO,EPCI,LIBEPCI,DATE_ARRETE,NB_VP_RECHARGEABLES_EL,NB_VP_RECHARGEABLES_GAZ,NB_VP
326550,02086,BICHANCOURT,200071785,CA Chauny-Tergnier-La Fère,2025-06-30,20,0,1110
499967,57509,NITTING,200068146,CC Sarrebourg Moselle Sud,2025-03-31,18,0,587
539536,91122,BURES-SUR-YVETTE,200056232,CA Communauté Paris-Saclay,2021-06-30,142,0,8677
420581,79134,GLÉNAY,247900798,CC du Thouarsais,2021-12-31,2,0,593
700872,91207,ÉGLY,200057859,CA Cur d'Essonne Agglomération,2024-09-30,125,0,6396


Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.

In [ ]:
print(f"Le dataset contient {df_ve_raw.shape[0]} lignes pour {df_ve_raw.shape[1]} variables.")

#### Revenus

In [4]:
print(f"Shape : {df_revenu_raw.shape}")
df_revenu_raw.sample(5)

Shape : (34926, 57)


,Nom géographique GMS,Code géographique,Libellé géographique,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Nbre d'unités de consommation dans les ménages fiscaux,[DISP] 1ᵉʳ quartile (€),[DISP] Médiane (€),[DISP] 3ᵉ quartile (€),[DISP] Écart interquartile (€),...,[DEC] 9ᵉ décile (€),[DEC] Rapport interdécile D9/D1,[DEC] S80/S20,[DEC] Iice de Gini,[DEC] Part des revenus d’activité (%),[DEC] dont part des salaires et traitements (%),[DEC] dont part des iemnités de chômage (%),[DEC] dont part des revenus des activités non salariées (%),"[DEC] Part des pensions, retraites et rentes (%)",[DEC] Part des autres revenus (%)
31814,Ambialet,81010,Ambialet,211.0,433.0,311.1,NaN,20940.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24073,Mencas,62565,Mencas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14145,Cognet,38116,Cognet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6130,Groises,18104,Groises,62.0,137.0,95.9,NaN,19400.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20041,Aulnois-en-Perthois,55015,Aulnois-en-Perthois,198.0,483.0,324.8,NaN,24010.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.

In [ ]:
print(f"Le dataset contient {df_revenu_raw.shape[0]} lignes pour {df_revenu_raw.shape[1]} variables.")

## 3. Nettoyage et Standardisation Géographique

Le défi majeur de ce projet est la fusion des données. 

#### Choix de la variable de jointure
Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus
La jointure sera faite sur ces variables.

#### Préparation des variables de jointure
Cependant les codes INSEE des communes sont souvent mal formatés.

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

#### Unification des codes
Nous utilisons la fonction standardize_all_codes pour garantir que chaque base dispose d'une colonne code_geo homogène sur 5 caractères.

In [5]:
dfs_to_clean = {
    "irve": (df_irve_raw, "code_insee_commune"),
    "revenu": (df_revenu_raw, "Code géographique"),
    "ve": (df_ve_raw, "CODGEO")
}

cleaned = standardize_all_codes(dfs_to_clean)
df_irve = cleaned["irve"]
df_revenu = cleaned["revenu"]
df_ve = cleaned["ve"]

Standardisation du code INSEE pour : irve (colonne : code_insee_commune)
Standardisation du code INSEE pour : revenu (colonne : Code géographique)
Standardisation du code INSEE pour : ve (colonne : CODGEO)


Une partie des données IRVE ne possède pas de code INSEE commune, mais dispose de coordonnées latitude/longitude. Plutôt que de supprimer ces lignes, nous utilisons un référentiel géographique pour retrouver la commune correspondante.
Grâce au croisement spatial avec les données de Cartiflette, nous avons pu attribuer un code commune à la quasi-totalité des points de charge.

In [6]:
# 1. Téléchargement des contours de communes via Cartiflette
communes_fr = charger_communes()

# 2. Transformation de l'IRVE en données géographiques (GeoDataFrame)
gdf_irve = creer_gdf_irve(df_irve, long_col="consolidated_longitude", lat_col="consolidated_latitude")

# 3. Jointure spatiale pour identifier la commune par les coordonnées GPS
gdf_result = joindre_communes(gdf_irve, communes_fr)

# 4. Intégration du code récupéré (code_geo_total) dans le dataframe principal
df_irve = ajouter_codes_geo(df_irve, gdf_result)

There was an error while reading the file from the URL: https://minio.lab.sspcloud.fr/projet-cartiflette/production/provider=IGN/dataset_family=ADMINEXPRESS/source=EXPRESS-COG-CARTO-TERRITOIRE/year=2022/administrative_level=COMMUNE/crs=4326/DEPARTEMENT=20/vectorfile_format=geojson/territory=metropole/simplification=0/raw.geojson
Error message: '/vsimem/pyogrio_30a8798b48c64c178a8a6262bf8ee077' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.


Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print({col: df_irve[col].isna().sum() for col in colonnes_irve})

print("\nValeurs uniques :")
print({col: df_irve[col].nunique() for col in colonnes_irve})

Valeurs manquantes :
{'code_insee_commune': np.int64(65645), 'code_geo_total': np.int64(1857)}

Valeurs uniques :
{'code_insee_commune': 9536, 'code_geo_total': 10317}


Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux. En effet, les données entrées n'étant pas toujours vérifiées, des erreurs étaient présentes. Certaines code géographiques correspondaient au code postal par exemple.

## 4. Feature Engineering : Passage à l'échelle communale

Réfléchissons aux variables à garder... Lesquelles pourraient être intéressante ?

In [8]:
list(df_irve.columns)

['nom_amenageur',
 'siren_amenageur',
 'contact_amenageur',
 'nom_operateur',
 'contact_operateur',
 'telephone_operateur',
 'nom_enseigne',
 'id_station_itinerance',
 'id_station_local',
 'nom_station',
 'implantation_station',
 'adresse_station',
 'code_insee_commune',
 'coordonneesXY',
 'nbre_pdc',
 'id_pdc_itinerance',
 'id_pdc_local',
 'puissance_nominale',
 'prise_type_ef',
 'prise_type_2',
 'prise_type_combo_ccs',
 'prise_type_chademo',
 'prise_type_autre',
 'gratuit',
 'paiement_acte',
 'paiement_cb',
 'paiement_autre',
 'tarification',
 'condition_acces',
 'reservation',
 'horaires',
 'accessibilite_pmr',
 'restriction_gabarit',
 'station_deux_roues',
 'raccordement',
 'num_pdl',
 'date_mise_en_service',
 'observations',
 'date_maj',
 'cable_t2_attache',
 'last_modified',
 'datagouv_dataset_id',
 'datagouv_resource_id',
 'datagouv_organization_or_owner',
 'created_at',
 'consolidated_longitude',
 'consolidated_latitude',
 'consolidated_code_postal',
 'consolidated_commune',
 '

In [9]:
list(df_ve.columns)

['CODGEO',
 'LIBGEO',
 'EPCI',
 'LIBEPCI',
 'DATE_ARRETE',
 'NB_VP_RECHARGEABLES_EL',
 'NB_VP_RECHARGEABLES_GAZ',
 'NB_VP',
 'code_geo']

In [10]:
list(df_revenu.columns)

['Nom géographique GMS',
 'Code géographique',
 'Libellé géographique',
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 "[DISP] Nbre d'unités de consommation dans les ménages fiscaux",
 '[DISP] 1ᵉʳ quartile (€)',
 '[DISP] Médiane (€)',
 '[DISP] 3ᵉ quartile (€)',
 '[DISP] Écart interquartile (€)',
 '[DISP] 1ᵉʳ décile (€)',
 '[DISP] 2ᵉ décile (€)',
 '[DISP]3ᵉ décile (€)',
 '[DISP] 4ᵉ décile (€)',
 '[DISP] 6ᵉ décile (€)',
 '[DISP] 7ᵉ décile (€)',
 '[DISP] 8ᵉ décile (€)',
 '[DISP] 9ᵉ décile (€)',
 '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile',
 '[DISP] S80/20',
 '[DISP] Iice de Gini',
 '[DISP] Part des revenus d’activité (%)',
 '[DISP] dont part des salaires et traitements(%)',
 '[DISP] dont part des iemnités de chômage (%)',
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des pensions, retraites et rentes (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)',
 "[DISP] Part de l'ensemble des prestat

Nos bases n'ont pas la même unité d'observation. L'IRVE liste des bornes (plusieurs par ville), alors que nous voulons une analyse par commune.

Pour pouvoir efectuer la jointure sur les codes géographiques, il est essentiel que pour chacune de nos 3 bases de données, chaque ligne corresponde à un unique code géographique.

Il faut alors réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.

Il faut également étudier la base df_ve afin de supprimer les "doublons" de code géographique.

#### IRVE
Ici, l'objectif est de mesurer l'offre de recharge par commune. Comme il peut y avoir plusieurs points de recharge par commune, il faut agréger. L'objectif final est d'expliquer ou prédire le taux de véhicules électriques local.

Après étude de notre base de données (cf etude_df_irve.ipynb) nous avons sélectionnés plusieurs variables nous permettant d'en créer de nouvelles, agrégées par code géographique.

Certaines variables ont été écarté directement car jugées non pertinente pour notre sujet, d'autres présentaient trop de valeurs manquantes, d'autres étaient trop peu informatives ou encore était du texte de libre donc trop difficile à utiliser avec le peu de temps dont nous disposons.

Fianleamnt voici un récap de la création des variables agrégées :

| Variables utilisées       | Variable finale                              | Construction        |
|--------------------------|-----------------------------------------------|---------------------|
| nbre_pdc                 | total_pdc                                     | somme               |
| puissance_nominale       | puissance_moyenne                             | moyenne             |
| puissance_nominale       | puissance_max                                 | max                 |
| nom_operateur            | nb_operateurs                                 | nunique             |
| nom_operateur            | top_operateur                                 | mode                |
| prise_type_2             | pct_type_2                                    | moyenne booléenne   |
| prise_type_combo_ccs     | pct_combo_ccs                                 | moyenne booléenne   |
| prise_type_chademo       | pct_chademo                                   | moyenne booléenne   |
| prise_type_ef            | pct_type_ef                                   | moyenne booléenne   |
| gratuit                  | pct_gratuit                                   | moyenne booléenne   |
| paiement_cb              | pct_paiement_cb                               | moyenne booléenne   |
| paiement_autre           | pct_paiement_autre                            | moyenne booléenne   |
| implantation_station     | prive, public, rapide, voirie                 | dummies + moyenne   |

Commençons par rendre les données des variables sélectionnées propres avant de faire l'agrégation.

En particulier, les données d'implantation des stations présentent des erreurs d'encodage (caractères spéciaux corrompus) et une trop grande diversité de labels. Nous procédons à une réparation du texte puis à un regroupement en 4 catégories majeures : privé, public, voirie et rapide.

In [11]:
# Traitement IRVE : Transformation des types et agrégation
df_irve = reparer_encodage_implantation(df_irve)
df_irve = regrouper_implantation_station(df_irve)
df_irve_clean = clean_irve_variables_finales(df_irve)
df_irve_final = creer_features_irve(df_irve_clean)
df_irve_final.sample(5)

,code_geo_total,total_pdc,puissance_moyenne,puissance_max,nb_operateurs,pct_type_2,pct_combo_ccs,pct_chademo,pct_type_ef,pct_gratuit,pct_paiement_cb,pct_paiement_autre,top_operateur,prive,public,rapide,voirie
4412,41194,450,80.252174,300.0,6,0.594203,0.333333,0.043478,0.405797,0.0,0.318841,1.0,Allego,0.710145,0.0,0.15942,0.130435
125,02141,32,36.040000,50.0,1,0.5,0.5,0.5,0.25,0.0,0.5,1.0,DRIVECO,0.0,1.0,0.0,0.0
9132,82048,4,23.000000,24.0,1,1.0,0.5,0.5,0.5,0.0,0.0,1.0,Syndicat Départemental d'Énergie de Tarn-et-Ga...,1.0,0.0,0.0,0.0
6684,62463,4,22.000000,22.0,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,Bouygues Energies & Services,0.0,0.0,0.0,1.0
8186,76617,23,14.806667,22.0,4,0.866667,0.0,0.333333,0.066667,0.0,0.333333,0.866667,Zephyre SAS,0.866667,0.0,0.0,0.133333


#### Véhicules
La base véhicules contient plusieurs dates d'observation pour une même commune.

Afin d'être cohérent avec les autres sources de données (IRVE, INSEE), nous souhaitons disposer d'une seule ligne par commune correspondant à la situation la plus récente disponible.

Nous conservons donc, pour chaque code commune (`CODGEO`), la dernière observation temporelle.

In [12]:
# Traitement VE : Sélection de la dernière observation
df_ve_final = garder_derniere_observation_commune(df_ve)
df_ve_final.head()

,CODGEO,LIBGEO,EPCI,LIBEPCI,DATE_ARRETE,NB_VP_RECHARGEABLES_EL,NB_VP_RECHARGEABLES_GAZ,NB_VP,code_geo
216883,01001,L'ABERGEMENT-CLÉMENCIAT,200069193,CC de la Dombes,2025-09-30,30,0,968,01001
432383,01002,L'ABERGEMENT-DE-VAREY,240100883,CC de la Plaine de l'Ain,2025-09-30,5,0,286,01002
541297,01004,AMBÉRIEU-EN-BUGEY,240100883,CC de la Plaine de l'Ain,2025-09-30,518,1,16080,01004
109071,01005,AMBÉRIEUX-EN-DOMBES,200042497,CC Dombes Saône Vallée,2025-09-30,68,0,2306,01005
216903,01006,AMBLÉON,200040350,CC Bugey Sud,2025-09-30,4,0,169,01006


À partir de cette base de données, nous allons construire la **variable cible** de notre analyse : le **taux de véhicules électriques par commune**.

Cette variable correspond à un ratio calculé à partir des variables suivantes :

- `NB_VP` : nombre total de voitures particulières ;
- `NB_VP_RECHARGEABLES_EL` : nombre de voitures particulières rechargeables / électriques.

Le taux est défini par la formule suivante :

$$
taux\_equipement\_ve = \frac{NB\_VP\_RECHARGEABLES\_EL}{NB\_VP}
$$

Ce ratio est plus pertinent que le nombre brut de véhicules électriques, car il permet de **neutraliser l’effet de taille des communes**.

En effet, une grande commune comptera naturellement davantage de véhicules électriques qu’une petite commune. Le recours à un taux permet donc de comparer plus justement le niveau d’équipement entre communes, quelle que soit leur population ou leur parc automobile total.

In [13]:
df_ve_final = creer_taux_equipement_ve(df_ve_final)
df_ve_final["taux_equipement_ve"].describe().round(4)

count    35197.0000
mean         0.0229
std          0.0182
min          0.0000
25%          0.0128
50%          0.0205
75%          0.0302
max          1.0000
Name: taux_equipement_ve, dtype: float64

Nous sélectionnons pour la modélisation les variables suivantes :
- CODGEO
- NB_VP
- NB_VP_RECHARGEABLES_EL
- taux_equipement_ve

In [14]:
vars_finales = [
        "CODGEO",
        "NB_VP",
        "NB_VP_RECHARGEABLES_EL",
        "taux_equipement_ve"
    ]
df_ve_final = df_ve_final[vars_finales]
df_ve_final.head(5)

,CODGEO,NB_VP,NB_VP_RECHARGEABLES_EL,taux_equipement_ve
216883,01001,968,30,0.030992
432383,01002,286,5,0.017483
541297,01004,16080,518,0.032214
109071,01005,2306,68,0.029488
216903,01006,169,4,0.023669


#### Revenus

Nous sélectionnons pour la modélisation les variables suivantes :
- Code géographique
- Libellé géographique
- [DISP] Médiane (€)
- [DISP] Iice de Gini
- [DISP] Nbre de ménages fiscaux
- [DISP] Nbre de personnes dans les ménages fiscaux
- [DISP] Part des revenus d’activité (%)
- [DISP] dont part des revenus des activités non salariées (%)
- [DISP] Part des revenus du patrimoine et autres revenus (%)

Pour plus de détails, veuillez vous référer au notebook « etude_df_revenu ».

In [15]:
var_selec = ['Code géographique',
'Libellé géographique',
'[DISP] Médiane (€)',
 '[DISP] Iice de Gini',
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 '[DISP] Part des revenus d’activité (%)',
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)'
]
df_revenu_final=df_revenu[var_selec]
df_revenu_final.sample(5)

,Code géographique,Libellé géographique,[DISP] Médiane (€),[DISP] Iice de Gini,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Part des revenus d’activité (%),[DISP] dont part des revenus des activités non salariées (%),[DISP] Part des revenus du patrimoine et autres revenus (%)
17451,48087,Prinsuéjols-Malbouzon,19770.0,NaN,123.0,242.0,NaN,NaN,NaN
12321,33005,Aernos-les-Bains,27410.0,0.281,6868.0,13748.0,55.3,6.2,14.1
23639,62119,Béthune,19290.0,0.308,11668.0,22953.0,68.9,6.1,7.0
18869,52125,Chamaraes-Choignes,26130.0,NaN,443.0,1029.0,NaN,NaN,NaN
6787,21199,Corsaint,22230.0,NaN,66.0,144.0,NaN,NaN,NaN


## 5. Jointure

Pour la base de données jointe nous avons choisis de garder les variables suivantes :
- IRVE
    - total_pdc
    - puissance_moyenne
    - puissance_max
    - nb_operateurs
    - top_operateur
    - pct_type_2
    - pct_combo_ccs
    - pct_chademo
    - pct_type_ef
    - pct_gratuit
    - pct_paiement_cb
    - pct_paiement_autre
    - prive
    - public
    - rapide
    - voirie          
- Véhicules
    - CODGEO
    - NB_VP
    - NB_VP_RECHARGEABLES_EL
    - taux_equipement_ve
- Revenus
    - Code géographique
    - Libellé géographique
    - [DISP] Médiane (€)
    - [DISP] Iice de Gini
    - [DISP] Nbre de ménages fiscaux
    - [DISP] Nbre de personnes dans les ménages fiscaux
    - [DISP] Part des revenus d’activité (%)
    - [DISP] dont part des revenus des activités non salariées (%)
    - [DISP] Part des revenus du patrimoine et autres revenus (%)

In [16]:
# Fusion des trois bases
df_final = (
    df_ve_final
    .merge(df_irve_final, left_on="CODGEO", right_on="code_geo_total", how="left")
    .merge(df_revenu_final, left_on="CODGEO", right_on="Code géographique", how="left")
)

print("Shape base finale :", df_final.shape)
df_final.sample(5)

Shape base finale : (35197, 30)


,CODGEO,NB_VP,NB_VP_RECHARGEABLES_EL,taux_equipement_ve,code_geo_total,total_pdc,puissance_moyenne,puissance_max,nb_operateurs,pct_type_2,...,voirie,Code géographique,Libellé géographique,[DISP] Médiane (€),[DISP] Iice de Gini,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Part des revenus d’activité (%),[DISP] dont part des revenus des activités non salariées (%),[DISP] Part des revenus du patrimoine et autres revenus (%)
3059,09237,32,0,0.000000,NaN,NaN,NaN,NaN,NaN,<NA>,...,<NA>,09237,Le Puch,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20204,55079,750,13,0.017333,NaN,NaN,NaN,NaN,NaN,<NA>,...,<NA>,55079,Brillon-en-Barrois,25310.0,NaN,282.0,710.0,NaN,NaN,NaN
11507,31174,728,13,0.017857,NaN,NaN,NaN,NaN,NaN,<NA>,...,<NA>,31174,Estadens,22220.0,NaN,246.0,508.0,NaN,NaN,NaN
2346,07317,1310,20,0.015267,NaN,NaN,NaN,NaN,NaN,<NA>,...,<NA>,07317,Talencieux,24410.0,NaN,448.0,1131.0,NaN,NaN,NaN
13298,35034,1458,24,0.016461,NaN,NaN,NaN,NaN,NaN,<NA>,...,<NA>,35034,La Boussac,21540.0,NaN,533.0,1236.0,NaN,NaN,NaN


## 6. Traitement des valeurs manquantes

In [17]:
df_final.isna().sum()

CODGEO                                                              0
NB_VP                                                               0
NB_VP_RECHARGEABLES_EL                                              0
taux_equipement_ve                                                  0
code_geo_total                                                  24880
total_pdc                                                       24880
puissance_moyenne                                               24880
puissance_max                                                   24880
nb_operateurs                                                   24880
pct_type_2                                                      24880
pct_combo_ccs                                                   24880
pct_chademo                                                     24880
pct_type_ef                                                     24880
pct_gratuit                                                     24906
pct_paiement_cb     

La base de données finale contient **35 197 communes**.

Parmi celles-ci, **24 919 communes ne disposent d’aucune donnée issue de la base IRVE**, soit environ **70 % de l’échantillon total**.

---

Après analyse du contexte territorial français, nous faisons l’hypothèse que ces communes peuvent être considérées comme **non équipées en points de recharge (IRVE)**.

Cette hypothèse est jugée réaliste dans le contexte suivant :

- faible densité de population ;
- activité économique limitée ;
- faible présence de parkings publics ;
- flux routiers réduits.

Dans ce type de communes, l’absence d’infrastructures de recharge est très fréquente et cohérente avec la structure territoriale française.

---

Les bornes de recharge sont principalement concentrées dans les zones suivantes :

- grandes villes et agglomérations ;
- axes routiers majeurs ;
- zones commerciales ;
- parkings publics structurants ;
- pôles intercommunaux centraux.

Ainsi, de nombreuses petites communes rurales ou périphériques ne disposent d’aucune infrastructure de recharge sur leur territoire.

---

Il est essentiel de noter qu’une absence locale de borne ne signifie pas nécessairement une absence d’accès à la recharge.
En effet une commune peut être dépourvue d’IRVE tout en étant située à quelques kilomètres d’une commune équipée, ce qui permet un accès indirect à la recharge.

---

L’absence de données IRVE pour une commune est donc interprétée comme une absence d’infrastructure locale

In [21]:
df_imputation = df_final.copy()

COLS_IRVE_NUMERIQUES = [
    'total_pdc', 
    'puissance_moyenne', 
    'puissance_max', 
    'nb_operateurs', 
    'pct_type_2', 
    'pct_combo_ccs', 
    'pct_chademo', 
    'pct_type_ef', 
    'pct_gratuit', 
    'pct_paiement_cb', 
    'pct_paiement_autre',
    'prive',
    'public',
    'rapide',
    'voirie'
]

COLS_IRVE_TEXTE = ['top_operateur']

df_imputation = imputer_valeurs_manquantes_fusion(df_imputation, cols_to_zero= COLS_IRVE_NUMERIQUES, cols_to_label=COLS_IRVE_TEXTE)
df_imputation.sample(5)

,CODGEO,NB_VP,NB_VP_RECHARGEABLES_EL,taux_equipement_ve,code_geo_total,total_pdc,puissance_moyenne,puissance_max,nb_operateurs,pct_type_2,...,voirie,Code géographique,Libellé géographique,[DISP] Médiane (€),[DISP] Iice de Gini,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Part des revenus d’activité (%),[DISP] dont part des revenus des activités non salariées (%),[DISP] Part des revenus du patrimoine et autres revenus (%)
9600,27251,694,18,0.025937,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,27251,Fontaine-l'Abbé,22590.0,NaN,231.0,536.0,NaN,NaN,NaN
34868,95011,485,7,0.014433,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,95011,Ambleville,28290.0,NaN,161.0,386.0,NaN,NaN,NaN
7862,23188,278,2,0.007194,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,23188,Saint-Dizier-les-Domaines,20500.0,NaN,101.0,192.0,NaN,NaN,NaN
8545,25111,443,10,0.022573,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,25111,Chalèze,25770.0,NaN,163.0,359.0,NaN,NaN,NaN
28660,72165,2349,58,0.024691,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,72165,Lombron,23220.0,NaN,798.0,1921.0,NaN,NaN,NaN


Nous pouvons désormais supprimer les variables `code_geo_total` et `Code géographique`.

Ces deux colonnes apportent une information redondante par rapport à `CODGEO`. Afin de simplifier la structure du jeu de données et d’éviter toute duplication inutile d’information, nous conservons uniquement la variable `CODGEO`, qui constitue l’identifiant géographique de référence.

In [22]:
df_imputation = df_imputation.drop(columns=["code_geo_total", "Code géographique"])

------------------------ début -------------

In [23]:
missing = pd.DataFrame({
    "nb_manquants": df_imputation.isna().sum(),
    "pct_manquants": (df_imputation.isna().mean() * 100).round(2)
})

missing.sort_values("nb_manquants", ascending=False)

,nb_manquants,pct_manquants
[DISP] Iice de Gini,29854,84.82
[DISP] Part des revenus du patrimoine et autres revenus (%),29854,84.82
[DISP] dont part des revenus des activités non salariées (%),29854,84.82
[DISP] Part des revenus d’activité (%),29854,84.82
top_operateur,24880,70.69
[DISP] Médiane (€),3876,11.01
[DISP] Nbre de personnes dans les ménages fiscaux,3876,11.01
[DISP] Nbre de ménages fiscaux,3876,11.01
Libellé géographique,275,0.78
CODGEO,0,0.00


[DISP] Iice de Gini
[DISP] dont part des revenus des activités non salariées (%)
[DISP] Part des revenus du patrimoine et autres revenus (%)
[DISP] Part des revenus d’activité (%)

Il va falloir enlever ces variables... trop de valeurs manquantes !

------------------------ fin -------------

## 7. Analyse des variables

A faire :
- analyse de la variable cible
- analyse descriptive des variables explicatives (univariées + bivariées)
- étude des corrélations
- nouveau tri des variables à la suite des analyses
- vérifier que le format des variables est correcte pour la modélisation.

## 8. Modélisation

Nous cherchons à prédire la variable taux_equipement_ve.

Approche 1 : Régression Lasso
Le modèle Lasso est idéal pour l'interprétabilité. Il permet de voir quelles variables "pèsent" réellement dans la décision d'achat d'un véhicule électrique.

Approche 2 : XGBoost
Ce modèle non-linéaire permet ...

Nos variables pour la modélisation sont les suivantes :

attention LASSO ne prend pas de valeurs manquantes !!

In [ ]:
features = ['total_pdc', 'puissance_max', 'nb_operateurs', 'pct_gratuit']
target = 'taux_equipement_ve' 

df_model = df_final[features + [target]].dropna()

# 2. Exécuter le Lasso (Baseline & Sélection)
model_l, metrics_l, coeffs_l = modelisation_lasso(df_model, features, target)
print(f"R2 Lasso : {metrics_l['r2']:.3f}")
print("Déterminants (Lasso) :\n", coeffs_l[coeffs_l != 0])

# 3. Exécuter le Gradient Boosting (Performance)
model_x, metrics_x, importance_x = modelisation_xgboost(df_final, features, target)
print(f"R2 XGBoost : {metrics_x['r2']:.3f}")
importance_x.plot(kind='barh', title="Importance des variables - XGBoost")

In [ ]:
features = ['[DISP] Médiane (€)', '[DISP] Iice de Gini', 'total_pdc', 'puissance_moyenne']
target = 'taux_equipement_ve'

model_lasso, metrics_lasso, coeffs = modelisation_lasso(df_master, features, target)
model_xgb, metrics_xgb, importance = modelisation_xgboost(df_master, features, target)

## 9. Analyse des résultats